# Wikibase Instance Diagnostics

**Project:** Tests  
**AI attribution:** GitHub Copilot (Claude Sonnet 4.6, 2026-07-29)

**Purpose:** Read-only checks that a Wikibase instance is correctly installed, configured, and populated — no bot credentials required.  
Edit **cell 2** to point at a different instance, then run all cells.

**No extra packages required** — uses Python stdlib only.

**Tests:**
- MediaWiki API reachable — siteinfo returns version and site name
- Wikibase extension installed — `WikibaseRepository` present in the extension list
- Item (Q) namespace configured — `Item` namespace registered in MediaWiki
- Property (P) namespace configured — `Property` namespace registered in MediaWiki
- Instance statistics — content pages, total pages, and user counts
- Read item Q1 — first item accessible via the Wikibase API
- Read property P1 — first property accessible, with datatype
- Full-text search — CirrusSearch returns results for a wildcard query
- SPARQL endpoint reachable — `ASK` query returns `true`
- SPARQL: labelled items exist — `COUNT` confirms at least one labelled entity

In [ ]:
# ── Target Wikibase instance ──────────────────────────────────────────────────
# Edit these values to point at a different wiki.
# They can also be set in ../.env  (WB_URL, SPARQL_URL)

WB_URL     = 'https://wikibase.kewl.org'       # Wikibase base URL (no trailing slash)
SPARQL_URL = 'https://query.kewl.org/sparql'   # SPARQL endpoint

# ── Load overrides from .env (stdlib only — no python-dotenv needed) ──────────
import os
from pathlib import Path

_env_file = Path('../.env')
if _env_file.exists():
    with open(_env_file) as _f:
        for _line in _f:
            _line = _line.strip()
            if _line and not _line.startswith('#') and '=' in _line:
                _k, _, _v = _line.partition('=')
                os.environ.setdefault(_k.strip(), _v.strip().strip('"\'')).rstrip('/')

WB_URL     = os.getenv('WB_URL',     WB_URL).rstrip('/')
SPARQL_URL = os.getenv('SPARQL_URL', SPARQL_URL).rstrip('/')
# Normalise SPARQL_URL: if it ends at the host only, append /sparql
if not SPARQL_URL.endswith('/sparql'):
    SPARQL_URL = SPARQL_URL + '/sparql'

print(f'Wikibase : {WB_URL}')
print(f'SPARQL   : {SPARQL_URL}')

In [ ]:
import json, urllib.request, urllib.parse

# ── Helpers ───────────────────────────────────────────────────────────────────
def _api(params):
    url = f'{WB_URL}/w/api.php?' + urllib.parse.urlencode({**params, 'format': 'json'})
    with urllib.request.urlopen(url, timeout=15) as r:
        return json.loads(r.read().decode())

def _sparql(query, timeout=15):
    params = urllib.parse.urlencode({'query': query, 'format': 'json'})
    req = urllib.request.Request(
        f'{SPARQL_URL}?{params}',
        headers={'Accept': 'application/sparql-results+json'},
    )
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

# ── Test runner ───────────────────────────────────────────────────────────────
_results = []

def _test(label, fn):
    try:
        detail = fn()
        _results.append((label, True))
        print(f'  PASS  {label}' + (f'  ->  {detail}' if detail else ''))
    except Exception as exc:
        _results.append((label, False))
        print(f'  FAIL  {label}  ->  {exc}')

# ── Tests ─────────────────────────────────────────────────────────────────────
_results.clear()
print(f'Instance diagnostics: {WB_URL}')
print('─' * 65)

# 1 — MediaWiki API reachable
def t1():
    r = _api({'action': 'query', 'meta': 'siteinfo', 'siprop': 'general'})
    g = r['query']['general']
    return f'{g["generator"]}  |  site: {g["sitename"]}'
_test('MediaWiki API reachable', t1)

# 2 — Wikibase extension installed
def t2():
    r = _api({'action': 'query', 'meta': 'siteinfo', 'siprop': 'extensions'})
    exts = {e.get('name', '') for e in r['query']['extensions']}
    wb_exts = sorted(e for e in exts if 'Wikibase' in e)
    assert wb_exts, f'no Wikibase extension found'
    return ', '.join(wb_exts)
_test('Wikibase extension installed', t2)

# 3 — Item namespace configured
def t3():
    r = _api({'action': 'query', 'meta': 'siteinfo', 'siprop': 'namespaces'})
    ns = r['query']['namespaces']
    found = next((v for v in ns.values()
                  if v.get('*') == 'Item' or v.get('canonical') == 'Item'), None)
    assert found, 'Item namespace not found'
    return f'namespace id={found["id"]}'
_test('Item (Q) namespace configured', t3)

# 4 — Property namespace configured
def t4():
    r = _api({'action': 'query', 'meta': 'siteinfo', 'siprop': 'namespaces'})
    ns = r['query']['namespaces']
    found = next((v for v in ns.values()
                  if v.get('*') == 'Property' or v.get('canonical') == 'Property'), None)
    assert found, 'Property namespace not found'
    return f'namespace id={found["id"]}'
_test('Property (P) namespace configured', t4)

# 5 — Statistics
def t5():
    r = _api({'action': 'query', 'meta': 'siteinfo', 'siprop': 'statistics'})
    s = r['query']['statistics']
    return (f'{s.get("articles", "?"):,} content pages  |  '
            f'{s.get("pages", "?"):,} total pages  |  '
            f'{s.get("users", "?"):,} users')
_test('Instance statistics', t5)

# 6 — Read Q1
def t6():
    r = _api({'action': 'wbgetentities', 'ids': 'Q1', 'props': 'labels|descriptions'})
    if 'error' in r:
        raise ValueError(r['error']['info'])
    e = r['entities'].get('Q1', {})
    if 'missing' in e:
        return 'Q1 does not exist yet'
    labels = {lang: v['value'] for lang, v in e.get('labels', {}).items()}
    return str(labels or '(no labels)')
_test('Read item Q1', t6)

# 7 — Read P1
def t7():
    r = _api({'action': 'wbgetentities', 'ids': 'P1', 'props': 'labels|datatype'})
    if 'error' in r:
        raise ValueError(r['error']['info'])
    e = r['entities'].get('P1', {})
    if 'missing' in e:
        return 'P1 does not exist yet'
    labels = {lang: v['value'] for lang, v in e.get('labels', {}).items()}
    return f'datatype={e.get("datatype", "?")}  labels={labels or "(none)"}'
_test('Read property P1', t7)

# 8 — Full-text search (CirrusSearch)
def t8():
    r = _api({'action': 'query', 'list': 'search',
              'srsearch': '*', 'srnamespace': '0|120|146', 'srlimit': '1'})
    if 'error' in r:
        raise ValueError(r['error'].get('info', r['error']))
    total = r.get('query', {}).get('searchinfo', {}).get('totalhits', 0)
    return f'{total:,} total hits'
_test('Full-text search (CirrusSearch)', t8)

# 9 — SPARQL endpoint
def t9():
    data = _sparql('ASK { ?s ?p ?o }', timeout=15)
    assert data.get('boolean') is True, f'unexpected response: {data}'
    return SPARQL_URL
_test('SPARQL endpoint reachable', t9)

# 10 — SPARQL: items with labels exist
def t10():
    data = _sparql(
        'SELECT (COUNT(?item) AS ?n) WHERE { ?item rdfs:label ?label }',
        timeout=30)
    n = int(data['results']['bindings'][0]['n']['value'])
    assert n > 0, 'no labelled items found via SPARQL'
    return f'{n:,} labelled item(s)'
_test('SPARQL: labelled items exist', t10)

# ── Summary ───────────────────────────────────────────────────────────────────
passed = sum(1 for _, ok in _results if ok)
total  = len(_results)
print('─' * 65)
verdict = 'ALL PASS ✓' if passed == total else f'FAILED {total - passed}/{total} ✗'
print(f'Result: {passed}/{total} passed  |  {verdict}')